# `optimize_from_mol` — Lead Optimisation with Fragment Operations

This notebook shows how to use `gen.optimize_from_mol()` to iteratively improve a **specific starting molecule** using LACAN's fragment operations (ring/linker/substituent replacement and scaffold decoration), guided by any user-defined scoring function.

Contrast with `generate_optimized_molecules()`, which starts from a random population. Use `optimize_from_mol` when you have:
- A known active or docking hit you want to elaborate
- A manually designed scaffold you want to optimise
- A lead series you want to explore systematically

---

**Contents**
1. Basic usage — LACAN score as objective
2. Optimise toward a molecular property (heavy atom count target)
3. Using a trained ML model as objective
4. Using the `callback` for live monitoring
5. Seeding `generate_optimized_molecules` from known actives
6. Protecting a pharmacophore while optimising the rest

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display
import matplotlib.pyplot as plt

from lacan import gen, lacan
from lacan import protect

# Load the ChEMBL LACAN profile (cached after first load)
profile = lacan.load_profile('chembl')
fragcorpus = gen.load_corpus(min_count=200)

print('Profile and corpus loaded.')

---
## 1. Basic Usage — LACAN Score as Objective

The simplest case: improve a molecule's LACAN score (chemical reasonableness).  
This is equivalent to what `mol_cleaner` does, but with fragment-level moves.

In [ ]:
# Seed molecule: a sildenafil-like core with a known bad bond environment
seed = Chem.MolFromSmiles('CCCc1nn(C)c2c(=O)[nH]c(-c3ccccc3)nc12')
seed_score, info = lacan.score_mol(seed, profile)
print(f'Seed LACAN score: {seed_score:.3f}, bad bonds: {info["bad_bonds"]}')

# Define the scoring function: must accept list[Mol] → list[float]
def lacan_score_fn(mols):
    return [lacan.score_mol(m, profile)[0] for m in mols]

# Run optimisation
results = gen.optimize_from_mol(
    seed,
    lacan_score_fn,
    profile,
    generations=10,
    beam_width=8,
    n_replacements=15,
    win_threshold=0.7,
    higher_is_better=True,
    conservative=True,
    fragcorpus=fragcorpus,
    plateau_patience=4,
)

print(f'\nFound {len(results)} results')
print('\nTop 5:')
for smi, score in results[:5]:
    print(f'  {score:.3f}  {smi}')

In [ ]:
# Draw the top results
top_mols = [Chem.MolFromSmiles(smi) for smi, _ in results[:9]]
labels = [f'{score:.3f}' for _, score in results[:9]]
display(Draw.MolsToGridImage(top_mols, molsPerRow=3, subImgSize=(300, 250),
                              legends=labels))

---
## 2. Optimise Toward a Molecular Property — Heavy Atom Count

Any `list[Mol] → list[float]` function works as the objective.  
Here we grow ibuprofen toward a target of 35 heavy atoms.

In [ ]:
ibuprofen = Chem.MolFromSmiles('CC(C)Cc1ccc(cc1)C(C)C(=O)O')
print(f'Ibuprofen: {ibuprofen.GetNumHeavyAtoms()} heavy atoms')

TARGET_SIZE = 35

def size_score_fn(mols):
    """Score = 1 - |n_heavy - target| / target. Max 1.0 at exact target."""
    return [max(0.0, 1.0 - abs(m.GetNumHeavyAtoms() - TARGET_SIZE) / TARGET_SIZE)
            for m in mols]

results = gen.optimize_from_mol(
    ibuprofen,
    size_score_fn,
    profile,
    generations=12,
    beam_width=8,
    n_replacements=20,
    win_threshold=0.85,       # within 15% of target size
    higher_is_better=True,
    fragcorpus=fragcorpus,
)

print(f'Found {len(results)} molecules near {TARGET_SIZE} heavy atoms')
top_mols = [Chem.MolFromSmiles(smi) for smi, _ in results[:9]]
sizes = [Chem.MolFromSmiles(smi).GetNumHeavyAtoms() for smi, _ in results[:9]]
labels = [f'score={sc:.2f} n={n}' for (_, sc), n in zip(results[:9], sizes)]
display(Draw.MolsToGridImage(top_mols, molsPerRow=3, subImgSize=(300, 250),
                              legends=labels))

---
## 3. Using a Trained ML Model as Objective

The `gen` module provides utilities to train a simple ECFP-based classifier on activity data. Here we use the D3 dopamine receptor dataset (included in LACAN's data directory) to score molecules for predicted D3 activity, then optimise a seed toward high predicted activity.

In [ ]:
import os
from rdkit.Chem import rdFingerprintGenerator
import numpy as np

# ─── Load D3 actives/decoys ───────────────────────────────────────────────
data_dir = os.path.join(os.path.dirname(gen.__file__), 'data')
actives_path = os.path.join(data_dir, 'd3_actives.csv')
decoys_path  = os.path.join(data_dir, 'd3_decoys.csv')

def load_smiles(path, col=0, has_header=True):
    mols = []
    with open(path) as f:
        for i, line in enumerate(f):
            if has_header and i == 0:
                continue
            smi = line.strip().split(',')[col]
            m = Chem.MolFromSmiles(smi)
            if m:
                mols.append(m)
    return mols

actives = load_smiles(actives_path)
decoys  = load_smiles(decoys_path)
print(f'Loaded {len(actives)} actives, {len(decoys)} decoys')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

MFPGEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def mols_to_fps(mols):
    return np.array([MFPGEN.GetFingerprintAsNumPy(m) for m in mols], dtype=float)

X = mols_to_fps(actives + decoys)
y = np.array([1]*len(actives) + [0]*len(decoys))

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='roc_auc')
print(f'5-fold CV ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

rf.fit(X, y)
print('Model trained.')

In [ ]:
# Build a scoring function from the trained model
def d3_activity_score(mols):
    """Returns predicted D3 activity probability (class 1 = active)."""
    fps = mols_to_fps(mols)
    return rf.predict_proba(fps)[:, 1].tolist()

# Seed: a known D3 active from the training set
seed = actives[0]
seed_score = d3_activity_score([seed])[0]
print(f'Seed predicted D3 activity: {seed_score:.3f}')
display(Draw.MolToImage(seed, size=(300, 200)))

# Optimise!
results = gen.optimize_from_mol(
    seed,
    d3_activity_score,
    profile,
    generations=15,
    beam_width=10,
    n_replacements=20,
    win_threshold=0.8,
    higher_is_better=True,
    conservative=True,
    fragcorpus=fragcorpus,
    plateau_patience=5,
)

print(f'\nFound {len(results)} winners with predicted activity ≥ 0.8')
top_mols = [Chem.MolFromSmiles(smi) for smi, _ in results[:9]]
labels = [f'p(active)={sc:.3f}' for _, sc in results[:9]]
display(Draw.MolsToGridImage(top_mols, molsPerRow=3, subImgSize=(300, 250), legends=labels))

---
## 4. Using the Callback for Live Monitoring

The `callback` receives a stats dict every generation.  Here we plot the score trajectory after the run.

In [ ]:
history = []

def record_stats(stats):
    history.append(stats)
    # Could also print live: print(f"Gen {stats['generation']}: best={stats['best_score']:.3f}")

results = gen.optimize_from_mol(
    ibuprofen,
    lacan_score_fn,
    profile,
    generations=15,
    beam_width=8,
    n_replacements=15,
    win_threshold=0.7,
    fragcorpus=fragcorpus,
    callback=record_stats,
    quiet=True,
)

# Plot trajectory
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
gens = [h['generation'] for h in history]

axes[0].plot(gens, [h['best_score'] for h in history], 'o-', color='steelblue')
axes[0].axhline(0.7, color='salmon', linestyle='--', label='win threshold')
axes[0].set_xlabel('Generation'); axes[0].set_ylabel('Best score')
axes[0].set_title('Score trajectory'); axes[0].legend()

axes[1].bar(gens, [h['n_candidates'] for h in history], color='steelblue', alpha=0.7, label='candidates')
ax2 = axes[1].twinx()
ax2.plot(gens, [h['n_winners'] for h in history], 's-', color='darkorange', label='cumulative winners')
axes[1].set_xlabel('Generation'); axes[1].set_ylabel('# candidates')
ax2.set_ylabel('# winners', color='darkorange'); ax2.tick_params(axis='y', labelcolor='darkorange')
axes[1].set_title('Candidates and winners per generation')

plt.tight_layout()
plt.show()

---
## 5. Seeding `generate_optimized_molecules` from Known Actives

When you have a set of known actives, you can use them as the starting population  
for the full GA via `seed_mols=`. The GA then crossbreeds, mutates, and decorates  
starting from your structures rather than random molecules.

Combine this with `bias_corpus` for maximum steering toward your chemotype.

In [ ]:
from lacan.gen import GAReporter

# Use the first 10 D3 actives as seed molecules
seed_mols = actives[:10]

# Optionally bias the fragment corpus toward the actives' chemistry
biased_corpus = gen.bias_corpus(seed_mols, fragcorpus=fragcorpus, ratio=3.0)
print(f'Biased corpus: {len(biased_corpus)} entries (vs {len(fragcorpus)} unbiased)')

reporter = GAReporter(label='D3 seed_mols + biased corpus')

winners = gen.generate_optimized_molecules(
    d3_activity_score,
    profile,
    seed_mols=seed_mols,          # ← start from known actives
    fragcorpus=biased_corpus,     # ← biased toward active chemistry
    generations=10,
    startN=20,
    popsize=15,
    win_threshold=0.75,
    higher_is_better=True,
    n_jobs=1,                     # set -1 for multiprocessing
    callback=reporter,
)

print(f'\nWinners found: {len(winners)}')
reporter.summary()

In [ ]:
reporter.plot()

In [ ]:
# Compare seeded vs random start
reporter_random = GAReporter(label='Random start')

gen.generate_optimized_molecules(
    d3_activity_score,
    profile,
    seed_mols=None,               # ← random start
    fragcorpus=fragcorpus,
    generations=10,
    startN=20,
    popsize=15,
    win_threshold=0.75,
    higher_is_better=True,
    n_jobs=1,
    callback=reporter_random,
)

GAReporter.compare([reporter, reporter_random], metric='best_pool')

---
## 6. Protecting a Pharmacophore While Optimising the Rest

Use atom protection to lock the pharmacophore (e.g. a sulfonamide warhead)  
and let `optimize_from_mol` only modify the rest of the molecule.

In [ ]:
from lacan.protect import protect_atoms_matching_smarts, protect_rejected_bonds

# Sildenafil with pyrimidinone core and a sulfonamide
sild = Chem.MolFromSmiles('CCCc1nn(C)c2c(=O)[nH]c(-c3cc(S(=O)(=O)N4CCN(C)CC4)ccc3OCC)nc12')

# Protect the sulfonamide pharmacophore — fragment operations will skip it
sild_protected = protect_atoms_matching_smarts(sild, 'S(=O)(=O)N')

# Also protect the bad bonds so they don't fail the LACAN filter
sild_protected = protect_rejected_bonds(sild_protected, profile)

initial_score, _ = lacan.score_mol(sild_protected, profile)
print(f'Protected sildenafil LACAN score: {initial_score:.3f}')

# Score function: LACAN score ignoring protected bonds
from lacan.protect import score_mol_ignoring_protected_bonds

def protected_lacan_score(mols):
    return [score_mol_ignoring_protected_bonds(m, profile)[0] for m in mols]

results = gen.optimize_from_mol(
    sild_protected,
    protected_lacan_score,
    profile,
    generations=10,
    beam_width=6,
    n_replacements=15,
    win_threshold=0.6,
    higher_is_better=True,
    conservative=True,
    fragcorpus=fragcorpus,
)

print(f'Found {len(results)} optimised variants')
# Show top 6 — sulfonamide should be present in all of them
top_mols = [Chem.MolFromSmiles(smi) for smi, _ in results[:6]]
labels = [f'{sc:.3f}' for _, sc in results[:6]]
display(Draw.MolsToGridImage(top_mols, molsPerRow=3, subImgSize=(300, 250), legends=labels))

# Verify sulfonamide is preserved
sulfonamide = Chem.MolFromSmarts('S(=O)(=O)N')
preserved = [Chem.MolFromSmiles(smi).HasSubstructMatch(sulfonamide) for smi, _ in results]
print(f'\nSulfonamide preserved in {sum(preserved)}/{len(preserved)} results')

---
## Summary

| Function | Use case |
|---|---|
| `optimize_from_mol(mol, fn, ...)` | Optimise a **specific lead** with fragment operations |
| `generate_optimized_molecules(fn, ..., seed_mols=[...])` | Full GA starting from **known structures** |
| `generate_optimized_molecules(fn, ...)` | Full GA from **random** starting population |

All three accept any `list[Mol] → list[float]` scoring function:  
LACAN score, ML model, docking wrapper, property calculator, or any combination.